# Package-EnvironmentLocal
- 価格や損益の情報を提供するパッケージ。以下の機能を実装する  
 - インポート  
     [Dependancy](#Dependency)
 - クラス変数  
     [クラス変数](#クラス変数)
 - MT5サーバから時間足情報を取得し、CSVファイルとして保存する  
    [MakePriceDataFile()](#クラス関数-MakePriceDataFile())
 - スタブ  
     [Main()](#Stub-Main())

## Dependency
[Page Top](#Package-Environment)  
[Page Top](https://colab.research.google.com/drive/1snkbEKO1hwqeLxwDsaGiSPfasDDQ46NY#scrollTo=buMqeeQBvg01&line=1&uniqifier=1)

In [5]:
from datetime import datetime,timezone,timedelta
#from dateutil.relativedelta import relativedelta
import pandas as pd
#import settings
import configparser
import os
import MetaTrader5 as mt5
from pathlib import Path

## Logger
ロギングを設定する。ログレベルは以下
1. CRITICAL
1. ERROR
1. WARNING
1. INFO
1. DEBUG

Environment package では、ログ空間を"DRLLogging.Environment"とする


[Page Top](#Package-Environment)

In [6]:
import logging
import logging.config

logconfigfile = configparser.ConfigParser()
logconfigfile.read('Logging.ini','UTF-8')
# logging.config.fileConfig('Logging_Local.ini')
logging.config.fileConfig(logconfigfile)
logger = logging.getLogger('DRLLogging')

## クラス変数
- MT5タイムフレーム変換辞書

[Page Top](#Package-Environment)

In [7]:
# .iniファイルを読み込む
inifile = configparser.ConfigParser()
inifile.read('settings.ini','UTF-8')

timrframe_dict = eval(inifile.get('LOCAL', 'timrframe_dict'))
# mt5_path = inifile.get('LOCAL', 'mt5_path')
mt5_real_path = inifile.get('LOCAL', 'mt5_real_path')
mt5_demo_path = inifile.get('LOCAL', 'mt5_demo_path')
#pricedata_path = inifile.get('LOCAL', 'pricedata_path')
pricedata_real_path = inifile.get('LOCAL', 'pricedata_real_path')
pricedata_demo_path = inifile.get('LOCAL', 'pricedata_demo_path')
#tickdata_path
tickdata_path_real = inifile.get('LOCAL', 'TICKDATA_PATH_REAL')
tickdata_path_demo = inifile.get('LOCAL', 'TICKDATA_PATH_DEMO')

## クラス関数 MakeRecentPriceDataFile()
- MT5サーバから時間足情報を取得し、CSVファイルとして保存する。関数を実行した時点におけるその年のISOWEEK１週目の月曜日から、最近の金曜日の最終足までのデータを取得する。
  - 引数：通貨ペア、時間足、上書きフラグ
  - 戻り値：なし
  - 入力：MT5プログラムのパスおよび実行ファイル名、CSVファイルの保存パス
  - 出力：時間足CSVファイル

In [8]:
def MakeRecentPriceDataFile(symbol, timeframe, rewriteflg,trdmd):
    # 関数実行時の今日の日時を取得する
    dt_today = datetime.now()
    start_day = datetime.fromisocalendar(dt_today.year,1,1)
            
    # ISOWEEKDAYは、月曜日=1、日曜日=7となる。月～金の時は終了日を先週の金曜日の23:59:00にする
    # 土日の場合は、終了日を今週の金曜日の23:59とする
    if dt_today.isoweekday() <= 5:
        end_day = datetime.fromisocalendar(dt_today.year,dt_today.isocalendar().week-1 ,6)-timedelta(minutes=1)
    else:
        end_day = datetime.fromisocalendar(dt_today.year,dt_today.isocalendar().week ,6)-timedelta(minutes=1)
        
    # realとdemoでパスを変える
    if trdmd == 0: # Demo口座の場合
        pricedata_path = pricedata_demo_path
    else: # Real口座の場合
        pricedata_path = pricedata_real_path
    
    # pricedata_filename = settings.pricedata_path+symbol+"_"+timeframe+"_"+start_day.strftime('%Y%m%d%H%M')+"_"+end_day.strftime('%Y%m%d%H%M')+".csv"
    pricedata_filename = pricedata_path+symbol+"_"+timeframe+"_"+start_day.strftime('%Y%m%d%H%M')+"_recent.csv"
    
    __GetPriceDataFromMT5(symbol, timeframe, rewriteflg,start_day,end_day,pricedata_filename,trdmd)
    return

## クラス関数 MakePriceDataFile()
- MT5サーバから時間足情報を取得し、CSVファイルとして保存する  
 - 引数：取得年、通貨ペア、時間足、上書きフラグ
 - 戻り値:なし
 - 入力：MT5プログラムのパスおよび実行ファイル名、CSVファイル保存パス
 - 出力:時間足CSVファイル 
 
- 注意：1分足や5分足を取得する場合は、あらかじめMT5クライアントの「チャートの最大バー数」を「Unlimited」にしておく  
    ツール→オプション→チャート  
 
[Page Top](#Package-Environment)


In [9]:
def MakePriceDataFile(year, symbol, timeframe, rewriteflg, trdmd):
    
    # ある年のisoweek1週目の月曜日(weekday=1)と最終週の日曜日(weekday=7)を取得
    # fromisocalendar()は、python3.8以降でないと実装されていない
    start_day = datetime.fromisocalendar(year,1,1)
    end_day = datetime.fromisocalendar(year+1,1,1)-timedelta(minutes=1)
    logger.debug("get price data from:%s to:%s",start_day,end_day)
    
    # realとdemoでパスを変える
    if trdmd == 0: # Demo口座の場合
        pricedata_path = pricedata_demo_path
    else: # Real口座の場合
        pricedata_path = pricedata_real_path
        
    # pricedata_filename = settings.pricedata_path+symbol+"_"+timeframe+"_"+start_day.strftime('%Y%m%d%H%M')+"_"+end_day.strftime('%Y%m%d%H%M')+".csv"
    pricedata_filename = pricedata_path+symbol+"_"+timeframe+"_"+start_day.strftime('%Y%m%d%H%M')+"_"+end_day.strftime('%Y%m%d%H%M')+".csv"
    
    __GetPriceDataFromMT5(symbol, timeframe, rewriteflg,start_day,end_day,pricedata_filename,trdmd)
    return
    
def __GetPriceDataFromMT5(symbol, timeframe, rewriteflg,start_day,end_day,pricedata_filename,trdmd):
    # timezoneをutcに設定する
    # 2023/4/10 修正
    # start_day.replace(tzinfo=timezone.utc)
    # end_day.replace(tzinfo=timezone.utc)
    start_day = start_day.replace(tzinfo=timezone.utc)
    end_day = end_day.replace(tzinfo=timezone.utc)
    
    # すでにファイルが取得されいて上書きしない場合は処理をしない
    if(not rewriteflg and os.path.isfile(pricedata_filename)):
        # print("%s is existing and not over-write." %(pricedata_filename))
        logger.warning("%s is existing and not over-write.", pricedata_filename)
        return
    
    logger.debug("%s starts criating.", pricedata_filename)
    # MT5クライアントを起動する
    # realとdemoでパスを分ける
    if trdmd == 0: # Demo口座の場合
        mt5_path = mt5_demo_path
    else: # Real口座の場合
        mt5_path = mt5_real_path
        
    # initialize(settings.mt5_path)
    logger.debug("MT5 initializing:%s",mt5_path)
    res = mt5.initialize(mt5_path)
    logger.debug("MT5 initialized result:%s", res)
    #時間足データを取得する(開始時間と終了時間 基準)
    rates = mt5.copy_rates_range(symbol, timrframe_dict[timeframe], start_day, end_day)
    # MT5クライアントをシャットダウンする
    mt5.shutdown()
    # 取得したデータはnumpy.ndarrayなので、pandasのdataframeに変換する
    df_rates = pd.DataFrame(rates)
    logger.debug("__GetPriceDataFromMT5:df_rates:%s", df_rates)
    # 秒での時間をdatetime形式に変換する
    df_rates['time']=pd.to_datetime(df_rates['time'], unit='s')
    # DataFrameをCSVファイルとして出力する
    df_rates.to_csv(pricedata_filename,index=False)
    
    logger.info("%s created.", pricedata_filename)
    return

## クラス関数 MakeTickDataFile()
- MT5サーバからティック情報を取得し、１か月ごとのCSVファイルとして保存する
 - 引数：取得年、通貨ペア、時間足、上書きフラグ
 - 戻り値:なし
 - 入力：MT5プログラムのパスおよび実行ファイル名、CSVファイル保存パス
 - 出力:月ごとのティックCSVファイル

In [17]:
def MakeTickDataFile(year, symbol, timeframe, rewriteflg,trdmd):

    # realとdemoでパスを分ける
    if trdmd == 0: # Demo口座の場合
        mt5_path = mt5_demo_path
    else: # Real口座の場合
        mt5_path = mt5_real_path    
    # MetaTrader 5ターミナルとの接続を確立する
    if not mt5.initialize(mt5_path):
        sys.exit("initialize() failed, error code =", last_error())
    
    # ローカルタイムゾーンオフセットの実装を回避するために、UTCタイムゾーンで「datetime」オブジェクトを作成する
    # ある年のISOCalendar１週目から最終週までのティックデータを取得し
    # 週ごとにCSVファイルとして保存する
    utc_from = datetime.fromisocalendar(year,1,1)
    utc_from = utc_from.replace(tzinfo=timezone.utc)
    utc_to = utc_from+timedelta(weeks=1)
    # (_,week_num,_) = (utc_from+timedelta(days=364)).isocalendar()
    # logger.info('Year %d has %d weeks.',year,week_num)
    (year_num,_,_) = utc_from.isocalendar()
    
    # realとdemoでパスを変える
    if trdmd == 0: # Demo口座の場合
        tickdata_path = tickdata_path_demo
    else: # Real口座の場合
        tickdata_path = tickdata_path_real

    # Tickを格納するフォルダが存在しない場合は、作成する
    dir = Path(tickdata_path+symbol)
    dir.mkdir(parents=True, exist_ok=True)

    # tickを取得しようとする週がisocalendar上当該年に属している間は取得する
    # for _ in range(week_num):
    while year_num == year:
        # MT5からTickを取得する
        tickdata_filename = tickdata_path+symbol+"\\"+symbol+"_"+timeframe+"_"+utc_from.strftime('%Y%m%d')+"_"+utc_to.strftime('%Y%m%d')+".csv"
        # すでにファイルが取得されいて上書きしない場合は処理をしない
        if(not rewriteflg and os.path.isfile(tickdata_filename)):
            logger.warning("%s is existing and not over-write.", tickdata_filename)
            # continue
        else:
            # １週間ごとのティックをリクエストする
            ticks = mt5.copy_ticks_range(symbol, utc_from, utc_to, mt5.COPY_TICKS_ALL)
            # 取得したデータはnumpy.ndarrayなので、pandasのdataframeに変換する
            df_tick = pd.DataFrame(ticks)
            # 秒での時間をdatetime形式に変換する
            df_tick['time']=pd.to_datetime(df_tick['time'], unit='s')
            df_tick['time_msc']=pd.to_datetime(df_tick['time_msc'], unit='ms')
    
            # DataFrameをCSVファイルとして出力する
            df_tick.to_csv(tickdata_filename,index=False)
            logger.info("%s created. %d ticks.", tickdata_filename,len(ticks))
        
        # １週間進める
        utc_from = utc_to
        utc_to += timedelta(weeks=1)
        # 次の週の属する年を取得する
        (year_num,_,_) = utc_from.isocalendar()
    
    # MT5クライアントをシャットダウンする
    mt5.shutdown()

## クラス関数 IteratedMakePriceDataFile()
- MakePriceDataFile() を繰り返し実行し、複数の通貨ペア、時間足、期間(年)の時間足CSVファイルを作成する
 - 引数：取得年[]、通貨ペア[]、時間足[]、上書きフラグ
 - 戻り値:なし
 - 入力：なし
 - 出力:なし

In [11]:
def IteratedMakePriceDataFile(years, symbols, timeframes, rewriteflg,trdmd):
    
    for y in years:
        for s in symbols:
            for tf in timeframes:
                if tf == 'TICK':
                    MakeTickDataFile(y, s, tf, rewriteflg,trdmd)
                else:
                    MakePriceDataFile(y, s, tf, rewriteflg,trdmd)
                    
    logger.info("■IteratedMakePiceDataFile Finished.")

In [12]:
def IteratedMakeRecentPriceDataFile(symbols, timeframes, rewriteflg,trdmd):
    
    for s in symbols:
        for tf in timeframes:
            if tf == 'TICK':
                # 実装していない
                # MakeTickDataFile(y, s, tf, rewriteflg)
                logger.warning("■IteratedMakePiceDataFile:MakeRecentTicedataFile not Implement.")
            else:
                MakeRecentPriceDataFile(s, tf, rewriteflg,trdmd)
                    
    logger.info("■IteratedMakePiceDataFile Finished.")

In [13]:
class PriceData:

  #@markdown ##Constractor <a name = "init"></a> 
  def __init__(self, sbl, prd):
    self.symbol = sbl
    self.period = prd
    # realとdemoでパスを変える
    if trdmd == 0: # Demo口座の場合
        self.pricedata_path = pricedata_demo_path
    else: # Real口座の場合
        self.pricedata_path = pricedata_real_path
        
    return
  
  def ConvertPriceDataFileToDataFrame(self):
    filenames=glob.glob('%s%s_%s*.csv'%(self.pricedata_path,self.symbol,self.period))
    logger.info(filenames)

    # ファイルリストを1つずつ取り出して、detaframeに加工する
    list_ = []
    for file in filenames:
      # csv(tsv)ファイルをpd.dataframeに変換
      df = pd.read_table(file)

      # <DATE>と<TIME>を結合し(間にスペースを挿入)、datetime型に変換
      # 新たな列<DATETIME>に結合結果を追加する
      df['<DATETIME>'] = pd.to_datetime(df['<DATE>'] + ' ' + df['<TIME>'], format='%Y.%m.%d %H:%M:%S')

      #<DATETIME>列をインデックスにした後に、<DATE>列と<TIME>列を削除する
      df = df.set_index('<DATETIME>')
      df = df.drop(['<DATE>','<TIME>'], axis=1)
      list_.append(df)

    price_list = pd.concat(list_)
    logger.info(price_list)
